In [50]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,datapointID,file_name,label
0,1,cc5aca4135cc4c7396bba1c32c9aa3f5.png,thorium
1,2,0bfd53ac789341ddaa3d3ea8c78833e7.png,thorium
2,3,580f87af75a742aeb20261b5163ba461.png,radium
3,4,6b1fa7669fc4417daf202b2fe1060d54.png,radium
4,5,e277e2554d264abe901542aa5585da03.png,radium


In [61]:
from PIL import Image
import numpy as np

def get_pixels(path):
    img = np.array(Image.open(path).convert("L"))
    cnt = 0
    
    for row in img:
        for pixel in row:
            if pixel != 0:
                cnt += 1

    return cnt

subtask1 = get_pixels("xrays.png")

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": [1],
    "answer": subtask1
})

df_1

,subtaskID,datapointID,answer
0,1,1,2393


In [62]:
from skimage.transform import hough_circle, hough_circle_peaks

img = np.array(Image.open("xrays.png").convert("L"))
hough_res = hough_circle(img, np.arange(20, 35, 2))
accums, cx, cy, radii = hough_circle_peaks(hough_res, np.arange(20, 35, 2), total_num_peaks=3)

accums

array([0.109375  , 0.10833333, 0.10833333])

In [63]:
def get_circles(path):
    img = np.array(Image.open(path).convert("L"))
    hough_res = hough_circle(img, np.arange(20, 35, 2))
    accums, cx, cy, radii = hough_circle_peaks(hough_res, np.arange(20, 35, 2), total_num_peaks=3)
    return accums

In [71]:
from tqdm import tqdm

train_circles = []
train_pixels = []

for fname in tqdm(df_train.file_name):
    path = f"train/{fname}"
    train_circles.append(get_circles(path))
    train_pixels.append(get_pixels(path))

test_circles = []
test_pixels = []

for fname in tqdm(df_test.file_name):
    path = f"test/{fname}"
    test_circles.append(get_circles(path))
    test_pixels.append(get_pixels(path))

pixels[:5]

100%|████████████████████████████████████████████████████████████████████████████████| 300/300 [03:01<00:00,  1.66it/s]


[1856, 5734, 1522, 4384, 1283]

In [74]:
X = pd.DataFrame(train_circles)
X["nr_pixels"] = train_pixels
y = df_train.label
X_final = pd.DataFrame(test_circles)
X_final["nr_pixels"] = test_pixels

X.columns = X.columns.astype(str)
X_final.columns = X_final.columns.astype(str)

X.head()

,0,1,2,nr_pixels
0,0.200000,0.191667,0.183333,3591
1,0.194444,0.191667,0.191667,2598
2,0.225000,0.203125,0.195312,1706
3,0.158333,0.156250,0.150000,1072
4,0.225000,0.203125,0.195312,1704


In [80]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=3, scoring='f1_macro').mean()

np.float64(0.8917293556828586)

In [81]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [82]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.datapointID,
    "answer": y_preds
})

df_2.head()

,subtaskID,datapointID,answer
0,2,1,radium
1,2,2,thorium
2,2,3,radium
3,2,4,thorium
4,2,5,radium


In [83]:
submission = pd.concat([df_1, df_2])
submission

,subtaskID,datapointID,answer
0,1,1,2393
0,2,1,radium
1,2,2,thorium
2,2,3,radium
3,2,4,thorium
...,...,...,...
295,2,296,xrays
296,2,297,radium
297,2,298,thorium
298,2,299,radium


In [84]:
submission.to_csv("submission.csv", index=False)